# Run the whole ladder on Kaggle — one session, leave it running

This notebook does end to end what `kaggle_stage_a` + `kaggle_stage_a_eval` +
`kaggle_merge_train` do piecemeal: **Stage A extraction → merge → eval bank →
the full ablation ladder → the robustness table**. Set `STREAM`, run all cells,
close the tab.

**It is resumable and it is phase-driven.** Every phase writes to
`/kaggle/working`, which survives the session as notebook output; every phase
checks whether its own output already exists and skips itself if so. A 12 hour
Kaggle session that dies in the middle of Stage A loses at most
`CHECKPOINT_EVERY` images — start a fresh session with the previous session's
output attached and it continues from where it stopped.

**Why `STREAM` and not a slug plus some flags.** The corpus and the
standardisation policy are not independent choices. The `coco_crop` dataset
must be extracted with `--canon-mode crop --crop-side 200 --geometric`;
extracting it under the default band policy is not an error and does not
produce an empty bank, it produces a *silently wrong* bank whose `config.json`
claims `mode="band"`. Tying both to one `STREAM` key makes that mistake
unspellable rather than merely documented.


## 0. Parameters

In [ ]:
# ============ THE ONLY LINES YOU NORMALLY EDIT ============
STREAM      = "union"       # "union" | "union_band" | "frozen" | "coco_crop"
                            # probes: "probe_crop" | "probe_band"
BACKBONE    = "siglip2l"    # siglip2l | dinov2l | convnextt | resnet50 |
                            # dinov3l (GATED, and float32 on Kaggle: its
                            # bfloat16 has no hardware on T4/P100, so
                            # run_dtype falls back and it runs ~3x slower.
                            # dinov2l is the same self-supervised lineage,
                            # Apache-2.0, ungated, and float16-safe here.)
SHARD_INDEX = 0             # which shard THIS session extracts
N_SHARDS    = None          # None = derive from the /kaggle/working quota
SMOKE       = True          # True first: proves the chain in minutes
PHASES      = "auto"        # "auto", or e.g. "stage_a" / "eval_bank,ladder"
# ==========================================================

# The corpus and its standardisation policy travel together. Splitting these
# into separate knobs is exactly how a band-mode bank gets built over
# crop-mode data, so they are one entry per stream and not four.
STREAMS = {
    "frozen": dict(
        slug="techjam-aigc-train",
        eval_slug="techjam-aigc-eval-manifest",
        canon_mode="band", crop_side=None, geometric=False,
    ),
    "coco_crop": dict(
        slug="techjam-aigc-train-coco-crop",
        eval_slug="techjam-aigc-eval-manifest-coco-crop",
        canon_mode="crop", crop_side=200, geometric=True,
    ),

    # ---- the union corpus -----------------------------------------------
    # Every licensed source in one frozen manifest: NTIRE, WildFake, SID_Set,
    # COCO train2017 and Open Images V7. 376,756 rows, balanced 188,319
    # authentic to 188,425 generated. This is the corpus the shipping detector
    # is trained on; `frozen` and `coco_crop` above are its two ancestors and
    # are kept only so their banks stay reproducible.
    #
    # TWO ENTRIES, ONE CORPUS, BECAUSE THE POLICY IS NOT DECIDED YET. Crop and
    # band standardisation are baked into the features at extraction, so the
    # choice cannot be revisited from a cached bank the way dataset
    # composition can. `probe_crop` / `probe_band` below answer it on 20,000
    # rows in ~90 minutes; run those FIRST, then launch the arm that won.
    # Both are spelled out here so the winner needs no edit, and the loser
    # should be deleted from this dict once the probe reports.
    #
    # `union` (crop) is listed first deliberately: it is the hypothesis, not
    # the default. Nothing about being first makes it right.
    #
    # geometric=False in both. Dihedral is a SEPARATE question from
    # standardisation and mixing them in the same run would leave neither
    # answered; it gets its own rung once the policy is settled.
    #
    # SHARDING. At dim 1024 the bank is 376,744 x 11 x 1024 float16 = 8.5 GiB,
    # comfortably inside the 20 GiB working quota -- so the quota is NOT the
    # binding constraint here, the 12 h session is. dinov2l runs at 518 px
    # (1,369 tokens) and is the expensive one; siglip2l at 384 px is roughly
    # half its cost. Do NOT guess the shard count: run with SMOKE=True first
    # and read `kb.session_plan`, which measures the rate on this session's
    # actual GPU and says how many sessions the shard needs.
    "union": dict(
        slug="techjam-aigc-union",
        eval_slug="techjam-aigc-eval-manifest-union",
        canon_mode="crop", crop_side=200, geometric=False,
    ),
    "union_band": dict(
        slug="techjam-aigc-union",
        eval_slug="techjam-aigc-eval-manifest-union",
        canon_mode="band", crop_side=None, geometric=False,
    ),

    # ---- the crop-vs-band A/B -------------------------------------------
    # These two DELIBERATELY break the rule the comment above states: same
    # corpus, two standardisation policies. That rule exists so nobody
    # accidentally extracts coco_crop data under the band policy; here it is
    # the entire experiment, so it is spelled out as its own stream rather
    # than reached by editing CANON_MODE by hand -- an edit that would leave
    # no trace of intent in the notebook.
    #
    # What makes it safe: `canon_policy` is recorded in every bank's
    # `extra_config`, so a band probe bank and a crop probe bank refuse to
    # merge, refuse to resume from each other and refuse to fuse. And both
    # run over a PROBE manifest, whose fingerprint differs from the real
    # coco_crop manifest, so neither can be mistaken for the shipping bank.
    #
    # Two things are held fixed on purpose:
    #   * geometric=False on BOTH arms. Dihedral augmentation needs a square
    #     input, so it is crop-only -- turning it on for the crop arm would
    #     test two changes at once and the result would not say which moved.
    #   * The same probe manifest and the same eval subsample for both arms,
    #     so the two banks cover identical rows and the comparison is a
    #     comparison of policy and nothing else.
    #
    # Sizing: 20,000 train+val rows (~45 min) and a 4,000-row eval subsample
    # (~35 min) per arm, against ~7 h + ~6 h for the full corpus. Run the two
    # arms as two concurrent sessions and the answer lands in ~1.5 h.
    #
    # Why the coco_crop corpus and not the frozen one: 85% of the frozen
    # corpus sits at exactly 200 px short side, where a 200x200 crop is
    # nearly the whole frame and the two policies collapse into each other.
    # coco_crop's images are 425-512 px, so the policies genuinely differ.
    # The probe rides on its OWN small Dataset -- 20,000 training images and
    # the eval subsample, ~10 GB -- not on the union's ~111 GB normalised
    # tree. Publishing 111 GB to answer a 90-minute question would cost more
    # than the question is worth, and `scripts/stage_manifest_images.py` cuts
    # exactly the images the probe manifests name.
    "probe_crop": dict(
        slug="techjam-aigc-union-probe",
        eval_slug="techjam-aigc-union-probe",
        canon_mode="crop", crop_side=200, geometric=False,
        manifest_glob="/kaggle/input/techjam-aigc-union-probe*/manifest_union_probe.parquet",
        eval_manifest_glob="/kaggle/input/techjam-aigc-union-probe*/eval_manifest_union_probe.parquet",
    ),
    "probe_band": dict(
        slug="techjam-aigc-union-probe",
        eval_slug="techjam-aigc-union-probe",
        canon_mode="band", crop_side=None, geometric=False,
        manifest_glob="/kaggle/input/techjam-aigc-union-probe*/manifest_union_probe.parquet",
        eval_manifest_glob="/kaggle/input/techjam-aigc-union-probe*/eval_manifest_union_probe.parquet",
    ),
}
assert STREAM in STREAMS, f"STREAM must be one of {sorted(STREAMS)}"
CFG = STREAMS[STREAM]

DATASET_SLUG   = CFG["slug"]
CANON_MODE     = CFG["canon_mode"]
CROP_SIDE      = CFG["crop_side"]
GEOMETRIC      = CFG["geometric"]
# Empty for the two full streams: they extract every eligible row.
EVAL_SUBSAMPLE = CFG.get("eval_subsample", {})

SPLITS = "train,val_internal"      # Stage B needs BOTH. Do not narrow this.
SEED   = 20260827                  # must match scripts/extract_features.py
TIER   = "ablation"

# The rungs that are runnable WITHOUT a reconstruction pass. a4 and a7 set
# use_recon and need recon.npy attached to both banks, which is a separate
# diffusers+LPIPS pass over the corpus and not part of this notebook.
RUNGS = ["a0", "a1", "a2", "a3", "a7_norecon"]

WORKERS          = 4
BATCH_SIZE       = 16
CHECKPOINT_EVERY = 200             # images between flushes = work at risk

REPO_URL = "https://github.com/bersamin12/robust-aigc-detection"
BRANCH   = "feat/robust-aigc-detection"
REPO_DIR = "/kaggle/working/robust-aigc-detection"

# A probe stream reads its manifest from a small separate Dataset -- the 64 GB
# image Dataset is shared with the full stream and is not re-uploaded for a
# 20,000-row cut. Note the probe Dataset's slug must NOT start with
# DATASET_SLUG, or DATA_GLOB below would sweep it in as an image mount.
MANIFEST_GLOB = CFG.get("manifest_glob",
                        f"/kaggle/input/{DATASET_SLUG}*/manifest.parquet")
DATA_GLOB     = f"/kaggle/input/{DATASET_SLUG}*"
EVAL_MANIFEST_GLOB = CFG.get(
    "eval_manifest_glob",
    f"/kaggle/input/{CFG['eval_slug']}*/eval_manifest*.parquet")
# The organisers' benchmark half of the eval manifest. Same Dataset for both
# streams: the benchmark is deliberately untouched so numbers stay comparable.
BENCH_MOUNT = "/kaggle/input/techjam-aigc-benchmark"

# Everything persists here or it does not persist. /kaggle/temp is wiped.
WORK       = "/kaggle/working"
BANK_DIR   = f"{WORK}/banks/{BACKBONE}"
SHARD_DIR  = f"{WORK}/banks/{BACKBONE}_shard{SHARD_INDEX}"
EVAL_DIR   = f"{WORK}/banks/eval_{BACKBONE}"
RUNS_DIR   = f"{WORK}/outputs/rungs"
DOCS_DIR   = f"{WORK}/outputs/docs"

print(f"stream={STREAM}  slug={DATASET_SLUG}")
print(f"  canon={CANON_MODE} crop_side={CROP_SIDE} geometric={GEOMETRIC}")
print(f"  backbone={BACKBONE}  shard={SHARD_INDEX}  smoke={SMOKE}")
if EVAL_SUBSAMPLE:
    print(f"  eval subsample={EVAL_SUBSAMPLE}")
    print(f"  PROBE stream: throwaway identity, not a shipping bank")
print(f"  rungs={RUNGS}")


## 1. Get the code

Public repo, shallow clone, and a `reset --hard` on re-run so a resumed session
picks up any fix without a stale working tree. Nothing here is authenticated.

In [ ]:
import glob, importlib, os, subprocess, sys, time

def sh(argv, **kw):
    """Run a command, show it, and fail loudly rather than continuing."""
    print("$", " ".join(str(a) for a in argv))
    return subprocess.run([str(a) for a in argv], check=True, **kw)

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    sh(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", BRANCH])
    sh(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"])
else:
    sh(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR])

# Two directories, for two different importers. `notebooks/` is where
# `kaggle_bootstrap` lives. `src/` is where the `aigcdet` package lives, and it
# is put on the path HERE rather than left to the `pip install -e` in the next
# section, because an editable install registers itself through a .pth file
# that site.py reads at INTERPRETER START. A kernel that was already running
# when pip finished never sees it. Every extraction is a subprocess with a
# fresh interpreter, so those work either way -- but an in-kernel
# `from aigcdet...` raises ModuleNotFoundError, several cells later, with the
# install cell reporting success.
for _sub in ("notebooks", "src"):
    _p = os.path.join(REPO_DIR, _sub)
    if _p not in sys.path:
        sys.path.insert(0, _p)
importlib.invalidate_caches()
import kaggle_bootstrap as kb
importlib.reload(kb)

sh(["git", "-C", REPO_DIR, "log", "--oneline", "-1"])
print("helper loaded from", kb.__file__)

## 2. Install — without losing Kaggle's torch

This is the step that ends sessions. `pip install -e .` hands pip the
`torch>=2.0` line from `pyproject.toml` and invites it to resolve a torch built
for a different CUDA than this machine's drivers; you get a `torch` that cannot
see the GPU and no way back except a factory reset.

So: the project goes in with `--no-deps` (a pure path registration, which is all
it is needed for), everything else is installed **only if genuinely missing**,
and nothing CUDA-matched is touched at all. `transformers` is the one exception
— Kaggle images move and the project needs ≥4.53 for DINOv3 — and it is
upgraded with `--no-deps` too.

**Print the plan before running it.** If you ever see `torch` in that list,
stop.

In [ ]:
def installed_version(dist):
    try:
        import importlib.metadata as im
        return im.version(dist)
    except Exception:
        return None

plan = kb.install_plan(os.path.join(REPO_DIR, "pyproject.toml"), REPO_DIR,
                       transformers_version=installed_version("transformers"))

print("pip plan:")
for cmd in plan:
    print("   ", " ".join(cmd))

assert not any(w.split("=")[0].split(">")[0] in ("torch", "torchvision", "triton")
               for cmd in plan for w in cmd), "STOP: the plan would touch torch"

for cmd in plan:
    sh(cmd, capture_output=True, text=True)
print("\ninstall done")

# Prove the project is importable IN THIS KERNEL, here, where the remedy is
# still "re-run the two cells above". Without this the first in-kernel
# `from aigcdet...` is in the auth section, and a ModuleNotFoundError there
# reads as an auth problem rather than a path one.
importlib.invalidate_caches()
from aigcdet.features.backbones import BACKBONES as _B
print("aigcdet importable:", len(_B), "backbones registered")

### 2b. Check the environment before paying for anything

Every problem this cell can report makes the 1.2 GB model download pointless,
so it runs before the download rather than after it.

If it tells you `transformers` was upgraded: **restart the kernel**
(Run → Restart session) and re-run from cell 0. A module already imported at
the old version stays imported.

In [ ]:
import platform

torch_v = installed_version("torch")
tf_v    = installed_version("transformers")
problems = kb.environment_problems(platform.python_version(), torch_v, tf_v)

print(f"python {platform.python_version()}  torch {torch_v}  transformers {tf_v}")
if problems:
    for p in problems:
        print("\nPROBLEM:", p)
    raise SystemExit("fix the above before continuing")

import torch
print("cuda available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
assert torch.cuda.is_available(), (
    "no GPU. Settings > Accelerator > GPU, then restart the session. "
    "Extraction on CPU will not finish inside a session.")

## 3. HuggingFace auth

**Only if your `BACKBONE` is gated.** SigLIP2 (Apache-2.0) and CLIP (MIT) are
public: the cell below will say so and move on, and you need no token at all.
DINOv3 is gated behind Meta's licence, and then two separate things must be
true — from this notebook they fail identically, as a 401/403 on
`from_pretrained`:

1. **Your own** HuggingFace account has accepted the licence at the model page.
   Acceptance is per account — the project owner's acceptance does nothing for
   yours.
2. A read token from that same account is attached to this notebook as a Kaggle
   Secret named `HF_TOKEN`.

**Never paste a token into a cell.** This repo is public and a notebook is
committed with its cell source. Add-ons → Secrets is the whole reason that
mechanism exists.

In [ ]:
# Both read off the registry, so they follow BACKBONE rather than being
# typed again here. DINOv3 is gated behind Meta's licence; SigLIP2
# (Apache-2.0) and CLIP (MIT) are not, and the fleet should not be
# stopped for a token its run never uses.
from aigcdet.features.backbones import BACKBONES
MODEL_ID = BACKBONES[BACKBONE].hf_id
GATED    = kb.requires_hf_token(BACKBONE)

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
except Exception:
    secrets = None

token = kb.hf_token(secrets)
for line in kb.hf_auth_advice(token, MODEL_ID, gated=GATED):
    print(line)

if token:
    # Exported for `transformers` to pick up. Never printed, never written to a
    # file that leaves this session.
    os.environ["HF_TOKEN"] = token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = token
elif GATED:
    raise SystemExit("no HuggingFace token -- see the instructions above")

## 4. Attach the data, and prove it is intact — **do not skip this**

Two things happen here.

**The mounts become one tree.** The manifest describes a single dataset root,
and Kaggle mounts each published Dataset at its own `/kaggle/input/<slug>/`. A
symlink farm in `/kaggle/temp` presents them as the one tree the manifest can be
rebased onto. Symlinks, not copies: the data is tens of GB and `/kaggle/input`
is read-only. `/kaggle/temp` is the right home for it precisely because it does
*not* persist — it costs nothing to rebuild next session.

**The files are checked against the frozen manifest.** `verify_images`
recomputes each file's digest and reports which of *missing* / *unreadable* /
*content-divergent* is wrong, because the fixes differ. This costs minutes.
Skipping it costs 8–13 GPU-hours of features that do not correspond to the
manifest's labels — and nothing downstream would tell you.

You cannot skip it by accident: the cell produces `GATE`, and every later cell
takes `GATE` as a required argument. No gate, no extraction.

> During a `SMOKE` run only a sample is digested, so it is quick. The real run
> digests everything. A clean sample is evidence, not proof, and the printout
> keeps saying so.

In [ ]:
import pandas as pd

MANIFEST = sorted(glob.glob(MANIFEST_GLOB))
DATA_MOUNTS = sorted(glob.glob(DATA_GLOB))
assert MANIFEST, f"no manifest Dataset attached (looked for {MANIFEST_GLOB})"
assert DATA_MOUNTS, f"no image Datasets attached (looked for {DATA_GLOB})"
assert len(MANIFEST) == 1, (
    f"{len(MANIFEST)} manifests match {MANIFEST_GLOB}: {MANIFEST}. The glob "
    "is a prefix, so DATASET_SLUG='techjam-aigc-train' also matches "
    "'techjam-aigc-train-coco-crop' -- two different corpora with different "
    "fingerprints. Picking one silently would extract a bank whose features "
    "come from a mix of both mounts. Detach the stream you are not running.")
MANIFEST = MANIFEST[0]
print(f"manifest: {MANIFEST}")
print(f"{len(DATA_MOUNTS)} image Dataset(s):")
for m in DATA_MOUNTS:
    print("   ", m)

# What the manifest says its root contains -- this is what locates the root
# inside each mount, instead of guessing at Kaggle's wrapper directories.
EXPECTED = kb.top_level_names(pd.read_parquet(MANIFEST, columns=["rel_path"]))
print("dataset root should contain:", sorted(EXPECTED))

UNIFIED = kb.unify_mounts(DATA_MOUNTS, "/kaggle/temp/aigcdet_root", EXPECTED)
DATA_ROOT = UNIFIED.root
print("unified root:", DATA_ROOT, "->", sorted(os.listdir(DATA_ROOT))[:8])

In [ ]:
t0 = time.time()
manifest, GATE = kb.open_verified_manifest(
    MANIFEST, DATA_ROOT,
    sample=2000 if SMOKE else None,
    # os.walk does not follow the farm's symlinked directories, so an
    # "extra files: 0" from it would be unearned. Skipped and said so.
    check_extra=not UNIFIED.linked,
)
print(kb.describe_gate(GATE))
print(f"\nverified in {time.time() - t0:.0f}s")
print(manifest["split"].value_counts().to_string())

## 5. What still needs doing

Each phase is skipped when its output already exists, so re-running this
notebook in a fresh session continues rather than restarts. `auto` runs every
phase that is not already complete.


In [ ]:
import json, os

def _bank_done(d):
    """A bank is complete when its metadata says every row was written."""
    st = kb.read_resume_state(d)
    return st.exists and st.n_images > 0 and st.n_done >= st.n_images

STATUS = {
    "stage_a":   _bank_done(SHARD_DIR),
    "merge":     _bank_done(BANK_DIR),
    "eval_bank": _bank_done(EVAL_DIR),
    "ladder":    os.path.exists(f"{DOCS_DIR}/robustness_table.md"),
}
ORDER = ["stage_a", "merge", "eval_bank", "ladder"]
TODO = ([p for p in ORDER if not STATUS[p]] if PHASES == "auto"
        else [p.strip() for p in PHASES.split(",") if p.strip()])

for p in ORDER:
    mark = "done" if STATUS[p] else ("TODO" if p in TODO else "skip")
    print(f"  {p:10s} {mark}")
print(f"\nthis session will run: {TODO or '(nothing)'}")

# Sharding: a bank that will not fit the 20 GiB working quota must be split,
# and guessing that number by hand is how a session dies at hour 11.
from aigcdet.features.backbones import BACKBONES
DIM = BACKBONES[BACKBONE].dim
n_rows = int(len(kb.select_splits(manifest, SPLITS)))
if N_SHARDS is None:
    # fits_in_working returns (ok, explanation) -- a TUPLE, so the shard count
    # has to be searched for rather than read off an attribute. Raising until
    # a shard fits uses only the public helper and cannot drift from it.
    N_SHARDS = 1
    while not kb.fits_in_working(-(-n_rows // N_SHARDS), DIM, n_views=11)[0]:
        N_SHARDS += 1
    print(f"\nN_SHARDS={N_SHARDS} (derived: {n_rows} rows x 11 views x dim {DIM})")
    print("  ", kb.fits_in_working(-(-n_rows // N_SHARDS), DIM, n_views=11)[1])
assert 0 <= SHARD_INDEX < N_SHARDS, (
    f"SHARD_INDEX={SHARD_INDEX} is outside 0..{N_SHARDS - 1}")


## 6. Phase 1 — Stage A

In [ ]:
if "stage_a" in TODO:
    plan_rows = kb.shard_plan(GATE, manifest, N_SHARDS, splits=SPLITS)
    mine = plan_rows[SHARD_INDEX]
    print(f"shard {SHARD_INDEX}/{N_SHARDS}: {mine['n_images']} images")

    argv = kb.run_shard_argv(
        GATE, manifest_path=MANIFEST, root=DATA_ROOT, backbone=BACKBONE,
        out_dir=SHARD_DIR, splits=SPLITS, shard=SHARD_INDEX,
        n_shards=N_SHARDS, resume=True, workers=WORKERS,
        batch_size=BATCH_SIZE, checkpoint_every=CHECKPOINT_EVERY,
        limit=64 if SMOKE else None,
        # The whole reason this notebook exists as one file: these three
        # arrive at extract_bank, are recorded in the bank config, and make
        # the bank refuse to merge or fuse with a differently-standardised one.
        canon_mode=CANON_MODE, crop_side=CROP_SIDE, geometric=GEOMETRIC)
    print(" ".join(argv[1:]), "\n")
    t0 = time.time()
    rc = kb.run_streaming(argv)
    assert rc == 0, f"stage A exited {rc}"
    elapsed = time.time() - t0
    print(f"\nstage A finished in {elapsed/60:.1f} min")

    if SMOKE:
        # The binding constraint is NOT the 20 GiB working quota -- at dim 1024
        # even the 182k-row corpus is a 3.9 GiB bank in one shard. It is the
        # 12 h session. Measure the rate on the smoke run and say up front how
        # many sessions the real run needs, rather than discovering it at hour
        # eleven.
        rate = kb.measure_rate(64, elapsed)
        plan = kb.session_plan(mine["n_images"], rate,
                               checkpoint_every=CHECKPOINT_EVERY)
        print(f"\n{rate:.3f} s/image (smoke, includes model download)")
        print(f"shard of {plan.n_images} images -> {plan.hours:.1f} h, "
              f"{plan.sessions_needed} session(s), "
              f"{plan.minutes_at_risk:.0f} min at risk per kill")
        for n in plan.notes:
            print("  !", n)
        print("\nSet SMOKE=False and re-run to start the real extraction.")
    else:
        st = kb.read_resume_state(SHARD_DIR)
        print(f"{st.n_done}/{st.n_images} images ({st.fraction_done:.1%})")
else:
    print("stage_a: skipped")


## 7. Phase 2 — merge

Skipped when `N_SHARDS == 1`: there is nothing to concatenate, so the shard
directory *is* the bank. Merging shards from other sessions requires them
attached as Datasets; `merge_banks` refuses a partial set rather than
silently producing a short bank.


In [ ]:
if "merge" in TODO:
    if N_SHARDS == 1:
        if not os.path.exists(BANK_DIR):
            os.makedirs(os.path.dirname(BANK_DIR), exist_ok=True)
            os.symlink(SHARD_DIR, BANK_DIR)
        print(f"one shard: {BANK_DIR} -> {SHARD_DIR}")
    else:
        shard_dirs = kb.sorted_shard_dirs(
            glob.glob(f"{WORK}/banks/{BACKBONE}_shard*")
            + glob.glob(f"/kaggle/input/aigcdet-bank-{BACKBONE}-shard*"))
        print(f"{len(shard_dirs)} shard(s):")
        for d in shard_dirs:
            print("   ", d)
        assert len(shard_dirs) == N_SHARDS, (
            f"expected {N_SHARDS} shards, found {len(shard_dirs)}. Attach the "
            "missing shard Datasets; a partial merge is refused, not silently "
            "truncated.")
        rc = kb.run_streaming(kb.merge_argv(BANK_DIR, shard_dirs, REPO_DIR))
        assert rc == 0, f"merge exited {rc}"
        print(kb.verify_merged_bank(BANK_DIR, MANIFEST, root=DATA_ROOT))
else:
    print("merge: skipped")


## 8. Phase 3 — eval bank

Note there is **no `--geometric`** here, deliberately. The eval grid scores one
image under 20 conditions, so all 20 must share a single standardisation: a
per-condition crop or orientation would confound "the score fell under
`jpeg_q30`" with "it was a different window", which is the one thing this bank
exists to measure.


In [ ]:
if "eval_bank" in TODO:
    ev = sorted(glob.glob(EVAL_MANIFEST_GLOB))
    assert ev, (
        f"no eval manifest attached (looked for {EVAL_MANIFEST_GLOB}). Publish "
        f"data/eval_manifest_{'' if STREAM=='frozen' else STREAM+'_'}*.parquet "
        f"as the Kaggle Dataset '{CFG['eval_slug']}' and attach it.")
    assert len(ev) == 1, f"{len(ev)} eval manifests match: {ev}"
    EVAL_MANIFEST = ev[0]

    # The eval manifest is rooted ONE LEVEL ABOVE the training root: its
    # rel_paths start with `demo/` (the organisers' benchmark) or with the
    # normalised tree's own name. DATA_ROOT is the inside of the latter, so
    # passing it here resolves nothing. Build the two-link farm instead, the
    # way kaggle_stage_a_eval does.
    import pandas as pd, shutil
    rel = pd.read_parquet(EVAL_MANIFEST, columns=["rel_path"])["rel_path"]
    expected = sorted({x.split("/")[0] for x in rel})
    train_names = [n for n in expected if n != "demo"]
    assert len(train_names) == 1 and "demo" in expected, (
        f"eval manifest top-level names are {expected}; this cell knows how to "
        "link exactly 'demo' plus one normalised tree.")
    assert os.path.isdir(BENCH_MOUNT), f"not attached: {BENCH_MOUNT}"

    EVAL_ROOT = "/kaggle/temp/aigcdet_eval_root"
    shutil.rmtree(EVAL_ROOT, ignore_errors=True)
    os.makedirs(EVAL_ROOT, exist_ok=True)
    os.symlink(DATA_ROOT, os.path.join(EVAL_ROOT, train_names[0]))
    os.symlink(BENCH_MOUNT, os.path.join(EVAL_ROOT, "demo"))
    print("eval root:", EVAL_ROOT, "->", sorted(os.listdir(EVAL_ROOT)))

    # A farm can list correctly and resolve to nothing when a mount came in
    # under a different slug. Prove it on real rows before an hour of GPU.
    missing = [x for x in rel.sample(200, random_state=SEED)
               if not os.path.exists(os.path.join(EVAL_ROOT, x))]
    assert not missing, (
        f"{len(missing)} of 200 sampled rows do not resolve, e.g. {missing[:3]}")
    print("200 sampled rows all resolve")

    argv = [sys.executable, f"{REPO_DIR}/scripts/extract_eval_bank.py",
            "--manifest", EVAL_MANIFEST, "--backbone", BACKBONE,
            "--out", EVAL_DIR, "--tier", TIER, "--root", EVAL_ROOT,
            "--device", "cuda", "--batch-size", str(BATCH_SIZE),
            "--checkpoint-every", str(CHECKPOINT_EVERY), "--resume",
            "--canon-mode", CANON_MODE]
    if CROP_SIDE is not None:
        argv += ["--crop-side", str(CROP_SIDE)]
    # A probe caps every split it scores. Both arms of the A/B pass the SAME
    # budgets and the same --subsample-seed, so the two eval banks cover
    # identical rows: the robustness curves differ by policy or not at all.
    for _split, _n in sorted(EVAL_SUBSAMPLE.items()):
        argv += ["--subsample", f"{_split}={_n}"]
    if EVAL_SUBSAMPLE:
        argv += ["--subsample-seed", str(SEED)]
    if SMOKE:
        argv += ["--limit", "64"]
    print(" ".join(argv[1:]), "\n")
    rc = kb.run_streaming(argv)
    assert rc == 0, f"eval bank exited {rc}"
else:
    print("eval_bank: skipped")


## 9. Phase 4 — the ladder

`run_ablation.py` trains every rung, scores each against the eval bank, writes
the robustness table and picks the headline by the §6.4 rule. It is itself
resumable: a rung whose checkpoint exists is not retrained, and every skip is
printed and recorded.


In [ ]:
if "ladder" in TODO:
    os.makedirs(DOCS_DIR, exist_ok=True)
    rung_cfgs = [f"{REPO_DIR}/configs/rungs/{r}.yaml" for r in RUNGS]
    for c in rung_cfgs:
        assert os.path.exists(c), f"missing rung config {c}"

    argv = [sys.executable, f"{REPO_DIR}/scripts/run_ablation.py",
            "--bank", BANK_DIR, "--eval-bank", EVAL_DIR,
            "--rungs", *rung_cfgs,
            "--tier", TIER, "--device", "cuda",
            "--out", f"{DOCS_DIR}/robustness_table.md",
            "--selection", f"{DOCS_DIR}/selection.json",
            "--heatmap", f"{DOCS_DIR}/robustness_heatmap.png",
            "--out-dir", RUNS_DIR]
    print(" ".join(argv[1:]), "\n")
    rc = kb.run_streaming(argv)
    assert rc == 0, f"ladder exited {rc}"
else:
    print("ladder: skipped")


## 10. What this session produced

In [ ]:
sel_path = f"{DOCS_DIR}/selection.json"
if os.path.exists(sel_path):
    sel = json.load(open(sel_path))
    print("headline:", sel.get("headline"))
    print("metric:  ", sel.get("metric"))
    for rung, v in sorted(sel.get("summary", {}).items()):
        print(f"  {rung:12s} {v.get('heldout_robust_tpr_at_1pct'):.4f}")
    print(f"\ntable: {DOCS_DIR}/robustness_table.md")
else:
    print("no ladder output yet -- run the remaining phases in a new session")

for d in (SHARD_DIR, BANK_DIR, EVAL_DIR):
    st = kb.read_resume_state(d)
    if st.exists:
        print(f"  {d}: {st.n_done}/{st.n_images} ({st.fraction_done:.0%})")

# Banks are big; publish them as Datasets so the next session can attach them
# instead of re-extracting. /kaggle/working survives as this notebook's output,
# but only until the next version overwrites it.
print(f"\npublish the bank as: aigcdet-bank-{BACKBONE}-shard{SHARD_INDEX}")


## The 2am playbook

Paste the failing error into the cell below and it will tell you whether a
re-run can possibly help. The short version:

**Retryable — re-run the cell as-is (with `RESUME=True`, nothing is repeated):**

* `CUDA out of memory` → lower `BATCH_SIZE` to 8, then 4.
* `ReadTimeout` / `ConnectionError` → the download or the clone; just re-run.
* `MemoryError` / the kernel dies → lower `WORKERS`.

**Fatal — re-running burns an hour of a 30 h weekly budget for nothing:**

* `gated repo` / `401` / `403` → your account has not accepted the DINOv3
  licence, or `HF_TOKEN` is not attached to *this* notebook.
* `cannot resume the bank at …` → the directory holds a **different** bank; a
  parameter moved between sessions, usually `SHARD_INDEX`. Restore the
  parameters it was started with, or extract to a new `OUT_DIR`. **Do not
  delete it** — that throws away completed images.
* `verify_images: FAILED` → the attached Datasets are not what the manifest was
  frozen against. The report names which of missing / unreadable / divergent it
  is; each has a different fix. Do not extract from this copy.
* `No space left on device` → the shard was always too big. Raise `N_SHARDS`.
* `bank has no val_internal rows` → `SPLITS` was narrowed. Re-extract.
* `libcudart` / `Torch not compiled with CUDA` → something replaced torch.
  **Do not pip install torch.** Factory-reset the session (Run → Factory reset)
  and start again.

**And never, whatever the error:**

* Do not run `scripts/build_dataset.py`. The manifest is frozen; re-splitting it
  after banks exist silently misaligns labels against features.
* Do not `reset_index()` a manifest frame. It re-keys every view's RNG.
* Do not merge a `--limit`ed smoke bank into the real one.

In [ ]:
# Paste the failing error (or the exception object) here.
ERROR = "CUDA out of memory. Tried to allocate 1.50 GiB"

print(kb.explain(ERROR))